# Inspect `cusip_ranks_v2__change_in_weight.parquet`

Quick description of the per-stock per-quarter rank parquet produced by `sweep_features_v2.py`.

In [1]:
from pathlib import Path
import pandas as pd

PATH = Path('cusip_ranks_v2__change_in_weight.parquet')
df = pd.read_parquet(PATH)
print(f'file:   {PATH}')
print(f'shape:  {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')

file:   cusip_ranks_v2__change_in_weight.parquet
shape:  139,121 rows x 6 cols
memory: 23.65 MB


## Schema (column types)

In [2]:
df.dtypes

cusip          object
year            int64
quarter         int64
mean_score    float32
rank            int64
model          object
dtype: object

## Head (first 10 rows)

In [3]:
df.head(10)

,cusip,year,quarter,mean_score,rank,model
0,487836108,2013,4,0.925773,1,WeightedLightGCN_v2
1,G6359F103,2013,4,0.923120,2,WeightedLightGCN_v2
2,88822Q103,2013,4,0.921839,3,WeightedLightGCN_v2
3,01449J105,2013,4,0.916148,4,WeightedLightGCN_v2
4,090911108,2013,4,0.910961,5,WeightedLightGCN_v2
5,478160104,2013,4,0.873597,6,WeightedLightGCN_v2
6,172967424,2013,4,0.873574,7,WeightedLightGCN_v2
7,443683107,2013,4,0.873150,8,WeightedLightGCN_v2
8,30231G102,2013,4,0.869363,9,WeightedLightGCN_v2
9,59156R108,2013,4,0.868874,10,WeightedLightGCN_v2


## Numeric summary

In [ ]:
df.describe(include='all').T

## Coverage: how many quarters and stocks

In [4]:
quarters = df.groupby(['year', 'quarter']).size().rename('n_stocks').reset_index()
print(f"unique quarters:    {len(quarters)}")
print(f"range:              {int(quarters['year'].min())}Q{int(quarters.iloc[0]['quarter'])} "
      f"-> {int(quarters['year'].max())}Q{int(quarters.iloc[-1]['quarter'])}")
print(f"unique CUSIPs:      {df['cusip'].nunique():,}")
if 'model' in df.columns:
    print(f"unique model tags:  {df['model'].unique().tolist()}")
quarters.head(15)

unique quarters:    47
range:              2013Q4 -> 2025Q2
unique CUSIPs:      4,398
unique model tags:  ['WeightedLightGCN_v2']


,year,quarter,n_stocks
0,2013,4,2544
1,2014,1,2637
2,2014,2,2701
3,2014,3,2768
4,2014,4,2815
5,2015,1,2857
6,2015,2,2864
7,2015,3,2889
8,2015,4,2903
9,2016,1,2938


## Top 10 stocks for the most recent quarter

In [ ]:
df['']

In [5]:
last_y, last_q = df.sort_values(['year', 'quarter']).iloc[-1][['year', 'quarter']]
last_y, last_q = int(last_y), int(last_q)
print(f'top 10 stocks for {last_y}Q{last_q}:')
(df[(df['year'] == last_y) & (df['quarter'] == last_q)]
   .sort_values('rank').head(10))

top 10 stocks for 2025Q2:


,cusip,year,quarter,mean_score,rank,model
136305,037833100,2025,2,1.000000,1,WeightedLightGCN_v2
136306,594918104,2025,2,1.000000,2,WeightedLightGCN_v2
136307,91324P102,2025,2,0.999997,3,WeightedLightGCN_v2
136308,46625H100,2025,2,0.999993,4,WeightedLightGCN_v2
136309,532457108,2025,2,0.999992,5,WeightedLightGCN_v2
136310,30231G102,2025,2,0.999974,6,WeightedLightGCN_v2
136311,478160104,2025,2,0.999968,7,WeightedLightGCN_v2
136312,742718109,2025,2,0.999959,8,WeightedLightGCN_v2
136313,166764100,2025,2,0.999948,9,WeightedLightGCN_v2
136314,437076102,2025,2,0.999937,10,WeightedLightGCN_v2


## What each column means

| Column | Meaning |
|---|---|
| `cusip` | Stock identifier |
| `year`, `quarter` | The quarter the model was TRAINED on |
| `mean_score` | Mean of `σ(fund_emb · stock_emb)` across all funds in that quarter — a stock's "average attractiveness" to the cohort of funds. Higher = more funds tend to touch this stock per the model |
| `rank` | 1 = highest `mean_score` in that quarter, len(stocks) = lowest |
| `model` | Tag identifying which sweep produced the row (e.g. `WeightedLightGCN_v2`) |

Note: in v2 these scores reflect Q's edges only (no out-of-sample prediction — that arrived in v3).